In [ ]:
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")
print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])

Connected to PostgreSQL: localhost storage_analytics


In [2]:
anomaly_df = pd.read_sql_query("SELECT * FROM anomaly_events", engine)
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)

curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"])
anomaly_df["timestamp"] = pd.to_datetime(anomaly_df["timestamp"])

anomaly_df.head()

,device,timestamp,metric_name,metric_value,detector_type,anomaly_score,severity,is_anomaly,details,source_file,...,util_pct,aqu_sz,avg_latency_ms,read_ratio,write_ratio,avg_request_size_kb,total_iops,total_throughput_mb_s,saturation_score,latency_pressure
0,dm-0,2026-03-30 02:42:22.614939+00:00,avg_latency_ms,2.256685,iqr,1.818107,low,1,"{""mean"": 1.0954345835112766, ""std"": 0.63871370...",data/raw/generated_iostat.csv,...,30.307912,0.366689,2.256685,0.276882,0.723118,75.714556,199.994516,14.787594,11.113579,0.827502
1,dm-0,2026-03-30 10:17:53.096655+00:00,avg_latency_ms,1.889496,iqr,1.243220,low,1,"{""mean"": 1.0954345835112766, ""std"": 0.63871370...",data/raw/generated_iostat.csv,...,20.960635,1.247892,1.889496,0.414553,0.585447,80.759906,563.107726,44.410671,26.156617,2.357888
2,dm-0,2026-03-30 13:07:41.917902+00:00,avg_latency_ms,1.826061,iqr,1.143903,low,1,"{""mean"": 1.0954345835112766, ""std"": 0.63871370...",data/raw/generated_iostat.csv,...,54.120505,1.797266,1.826061,0.447359,0.552641,102.443938,619.837935,62.010390,97.268940,3.281918
3,dm-0,2026-03-30 16:12:18.358062+00:00,avg_latency_ms,1.773905,iqr,1.062244,low,1,"{""mean"": 1.0954345835112766, ""std"": 0.63871370...",data/raw/generated_iostat.csv,...,33.200890,1.107113,1.773905,0.487759,0.512241,67.248696,638.360285,41.922751,36.757126,1.963912
4,dm-0,2026-03-30 18:32:38.806525+00:00,avg_latency_ms,1.967627,iqr,1.365546,low,1,"{""mean"": 1.0954345835112766, ""std"": 0.63871370...",data/raw/generated_iostat.csv,...,35.913830,0.609445,1.967627,0.469875,0.530125,84.172677,394.945726,32.464491,21.887504,1.199161


In [3]:
anomaly_df["detector_type"].value_counts()
anomaly_df["severity"].value_counts()
anomaly_df["metric_name"].value_counts()

metric_name
saturation_score         784
iowait_pressure          574
avg_latency_ms           479
queue_efficiency         434
total_iops               277
merge_efficiency         267
total_throughput_mb_s     75
Name: count, dtype: int64

In [4]:
anomaly_df.groupby("device").size().sort_values(ascending=False)

device
sdb        740
sda        589
nvme0n1    561
dm-0       500
nvme1n1    500
dtype: int64

In [5]:
context_cols = [
    "device", "timestamp", "merge_efficiency", "queue_efficiency",
    "iowait_pressure", "svctm_await_ratio", "write_amplification",
    "workload_pattern", "saturation_score",
 ]
joined = anomaly_df.merge(curated_df[context_cols], on=["device", "timestamp"], how="left")
joined.sort_values("anomaly_score", ascending=False).head(20)

,device,timestamp,metric_name,metric_value,detector_type,anomaly_score,severity,is_anomaly,details,source_file,...,total_throughput_mb_s,saturation_score_x,latency_pressure,merge_efficiency,queue_efficiency,iowait_pressure,svctm_await_ratio,write_amplification,workload_pattern_y,saturation_score_y
2412,sdb,2026-04-05 20:47:13.904105+00:00,avg_latency_ms,65239.601770,zscore+iqr,36.022138,critical,1,"{""mean"": 255.61360058357138, ""std"": 1804.00140...",data/raw/generated_iostat.csv,...,4.231514,12.466960,13932.598337,0.080408,853.281290,7.359363,0.000087,0.946982,latency_sensitive,12.466960
2565,sdb,2026-04-03 19:17:46.883010+00:00,saturation_score,8789.858007,zscore+iqr,29.260770,critical,1,"{""mean"": 140.77213694744236, ""std"": 295.586402...",data/raw/generated_iostat.csv,...,9.857444,8789.858007,584926.396154,0.161953,4.121025,32.035633,0.002107,2.437555,latency_sensitive,8789.858007
229,dm-0,2026-04-05 17:42:00.013531+00:00,saturation_score,1704.456553,zscore+iqr,25.262961,critical,1,"{""mean"": 35.00582211755865, ""std"": 66.08293914...",data/raw/generated_iostat.csv,...,20.119877,1704.456553,185.126500,0.093079,10.107286,23.198580,0.167934,1.381642,latency_sensitive,1704.456553
1069,nvme1n1,2026-03-31 09:12:38.671256+00:00,avg_latency_ms,7.628814,zscore+iqr,24.382387,critical,1,"{""mean"": 0.31695923895303657, ""std"": 0.2998826...",data/raw/generated_iostat.csv,...,74.528292,63.793122,15.266852,0.045253,579.719258,4.688775,0.185215,0.848520,latency_sensitive,63.793122
1602,sda,2026-04-05 01:17:17.628741+00:00,avg_latency_ms,192.610421,zscore+iqr,22.498915,critical,1,"{""mean"": 4.380380059506749, ""std"": 8.366183023...",data/raw/generated_iostat.csv,...,11.050206,4.362290,29.414950,0.015030,5023.592375,0.000000,0.008660,6.645866,latency_sensitive,4.362290
1068,nvme1n1,2026-03-31 09:07:52.214182+00:00,avg_latency_ms,6.431270,zscore+iqr,20.389012,critical,1,"{""mean"": 0.31695923895303657, ""std"": 0.2998826...",data/raw/generated_iostat.csv,...,61.063840,72.874932,26.750433,0.024352,243.750372,0.000000,0.229765,0.493725,latency_sensitive,72.874932
1070,nvme1n1,2026-03-31 09:17:15.966970+00:00,avg_latency_ms,6.421059,zscore+iqr,20.354963,critical,1,"{""mean"": 0.31695923895303657, ""std"": 0.2998826...",data/raw/generated_iostat.csv,...,73.156218,130.165748,20.434639,0.034826,312.355151,2.222489,0.228339,0.612243,latency_sensitive,130.165748
52,dm-0,2026-04-05 17:52:31.461984+00:00,avg_latency_ms,13.035044,zscore+iqr,18.693210,critical,1,"{""mean"": 1.0954345835112766, ""std"": 0.63871370...",data/raw/generated_iostat.csv,...,13.359289,742.156454,104.318612,0.056616,24.471656,22.248917,0.156708,1.045919,latency_sensitive,742.156454
474,dm-0,2026-04-05 18:02:06.266171+00:00,iowait_pressure,34.350840,zscore+iqr,18.484828,critical,1,"{""mean"": 0.8789368920974464, ""std"": 1.81077711...",data/raw/generated_iostat.csv,...,15.673463,416.329227,55.556203,0.058592,49.058230,34.350840,0.125577,1.080299,latency_sensitive,416.329227
471,dm-0,2026-04-05 17:47:44.925140+00:00,iowait_pressure,34.219611,zscore+iqr,18.412357,critical,1,"{""mean"": 0.8789368920974464, ""std"": 1.81077711...",data/raw/generated_iostat.csv,...,15.202780,528.253449,36.684209,0.098509,45.869754,34.219611,0.193488,0.717058,latency_sensitive,528.253449


In [6]:
focus_metrics = ["iowait_pressure", "merge_efficiency", "queue_efficiency", "avg_latency_ms", "saturation_score"]
metric_workload = anomaly_df.groupby(["metric_name", "workload_pattern"]).size().sort_values(ascending=False).head(20)
focus_severity = (
    anomaly_df[anomaly_df["metric_name"].isin(focus_metrics)]
    .groupby(["metric_name", "severity"]).size()
    .unstack(fill_value=0)
    .sort_index()
)
{"metric_workload_top20": metric_workload, "focus_metric_severity": focus_severity}

{'metric_workload_top20': metric_name       workload_pattern 
 avg_latency_ms    latency_sensitive    376
 saturation_score  balanced             238
                   burst_io             236
 iowait_pressure   burst_io             233
 merge_efficiency  balanced             229
 queue_efficiency  balanced             188
 saturation_score  latency_sensitive    178
 queue_efficiency  latency_sensitive    164
 iowait_pressure   balanced             126
                   saturated            115
 saturation_score  saturated            115
 total_iops        latency_sensitive    107
                   burst_io              96
 iowait_pressure   latency_sensitive     73
 avg_latency_ms    balanced              67
 queue_efficiency  burst_io              44
 total_iops        saturated             44
 avg_latency_ms    burst_io              34
 merge_efficiency  latency_sensitive     31
 queue_efficiency  small_io_pressure     30
 dtype: int64,
 'focus_metric_severity': severity         